# Google Maps Scraper

A reusable Google Maps scraper that works with **any keyword and any city/country** — no code changes required.

It extracts **name, address, rating, and phone number** for every place in the search results and exports them to Excel.

**How to use:** run the cells in order, set your `keyword` and `city` in Cell 3, then run the rest.

**Tech:** Python · Selenium · pandas · openpyxl

## Cell 1 — Install dependencies
Run once on a new machine. Skip if the libraries are already installed.

In [ ]:
# Install the libraries this scraper needs:
#   selenium          = drives Chrome automatically
#   webdriver-manager = downloads the matching ChromeDriver for you (no manual download)
#   openpyxl          = writes the Excel file
!pip install selenium webdriver-manager openpyxl

## Cell 2 — Imports and setup

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait          # wait for an element with a condition
from selenium.webdriver.support import expected_conditions as EC # ready-made conditions, e.g. "element is present"
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random
import re

print("✅ Imports loaded")

## Cell 3 — Set your keyword and city

**Edit only this cell** when you want a new search.

Examples:
- `keyword = "halal food"`, `city = "tokyo"`
- `keyword = "hotel"`, `city = "bangkok"`
- `keyword = "coffee shop"`, `city = "paris"`
- `keyword = "dentist"`, `city = "new york"`

In [ ]:
# ============================
#  Edit only these 3 lines
# ============================

keyword      = "halal food"   # type of place to search for
city         = "tokyo"        # city or country
scroll_times = 10             # how many times to scroll down to load more places (increase for more results)

# ============================

output_filename = f"{keyword.replace(' ', '_')}_{city.replace(' ', '_')}.xlsx"
print(f"🔍 Search: '{keyword}' in '{city}'")
print(f"💾 Output file: {output_filename}")

## Cell 4 — Helper functions
Reusable logic kept in functions so the main scraper stays readable.

The parsing logic is split into **pure functions** (string in → string out, no browser) so it can be unit-tested without opening Chrome — see Cell 4.5.

In [ ]:
def create_driver():
    """
    Create a Chrome driver tuned to look more human
    (reduces the chance Google blocks the automation).
    """
    options = Options()
    # Hide the flag that says the browser is automated
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    # Use a normal-looking Chrome user-agent
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    # Hide that the browser is controlled by Selenium
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def scroll_to_load_results(driver, times):
    """
    Scroll the results list down to load more places.
    Google Maps lazy-loads results, so we must scroll before they all appear.
    """
    try:
        scrollable = driver.find_element(By.CSS_SELECTOR, "div[role='feed']")
        for i in range(times):
            driver.execute_script(
                "arguments[0].scrollTop = arguments[0].scrollHeight", scrollable
            )
            time.sleep(random.uniform(1.5, 2.5))  # wait for the new batch to load
        print(f"   scrolled {times} times")
    except Exception as e:
        print(f"   ⚠️ could not scroll: {e}")


def wait_for_place_details(driver, timeout=12):
    """
    Wait until the place detail page has loaded, detected by the place title (h1).
    This replaces guessing with sleep() — fast pages continue immediately,
    slow pages (e.g. hotels with a booking panel) get the time they need.
    Returns True if it loaded in time, False on timeout.
    """
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
        )
        return True
    except Exception:
        return False


# ======================================================================
#  Pure functions — string in, string out, no browser involved.
#  Split out so they can be unit-tested without opening Chrome (see Cell 4.5).
#  get_phone / get_address / get_rating delegate the cleaning to these.
# ======================================================================

def clean_phone_label(raw: str) -> str:
    """Strip a leading label such as 'Phone: ' or 'Telefon: ' off the number."""
    return re.sub(r"^[^:]+:\s*", "", raw).strip()


def clean_address_text(raw: str) -> str:
    """
    Remove only the 'junk' characters that are not part of a real address, by codepoint.
    Filtering by ord() (ASCII-hex only) keeps the source robust — there are no raw
    special characters in the code that an editor could silently corrupt.

    Dropped ranges:
      >= 0x10000          emoji in the supplementary planes
      0x2600 - 0x27BF     miscellaneous symbols + dingbats
      0xE000 - 0xF8FF     Private Use Area — the map-pin icon font (renders as a box □)
      0xFE00 - 0xFE0F     variation selectors (invisible emoji modifiers)

    Real address characters are kept, e.g. the Japanese postal mark 〒 (U+3012)
    and the hyphen in postal codes like 150-0041.
    """
    def keep(ch):
        o = ord(ch)
        if o >= 0x10000:          return False
        if 0x2600 <= o <= 0x27BF: return False
        if 0xE000 <= o <= 0xF8FF: return False
        if 0xFE00 <= o <= 0xFE0F: return False
        return True
    cleaned = "".join(ch for ch in raw if keep(ch))
    # Collapse all newlines/repeated whitespace into a single space, then trim
    return re.sub(r"\s+", " ", cleaned).strip()


def parse_rating(raw: str) -> str:
    """Pull the first numeric rating out of an aria-label. Return 'N/A' if none."""
    match = re.search(r"[\d.]+", raw)
    return match.group() if match else "N/A"


# ======================================================================
#  Browser-facing functions — wait for the data, then hand the raw
#  text to the pure functions above.
# ======================================================================

def get_phone(driver, timeout=5):
    """
    Get the phone number from a place detail page — waits up to `timeout` seconds.
    Returns 'N/A' if the element never appears (the place genuinely has no phone).
    """
    try:
        btn = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-item-id^='phone']"))
        )
        return clean_phone_label(btn.get_attribute("aria-label"))
    except Exception:
        return "N/A"


def get_address(driver, timeout=8):
    """
    Get the address from a place detail page.

    Waits up to `timeout` seconds for the address TEXT to actually populate — not just
    for the element to exist. This is the key reliability fix: on slow pages such as
    hotels, the address element appears empty first and the text fills in a moment later,
    so waiting only for "element present" would read a blank string.

    Works in every country — no country-specific trimming.
    Returns 'N/A' if no address text appears in time.
    """
    sel = "[data-item-id='address']"
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.find_element(By.CSS_SELECTOR, sel).text.strip() != ""
        )
        return clean_address_text(driver.find_element(By.CSS_SELECTOR, sel).text)
    except Exception:
        return "N/A"


def get_rating(element):
    """Get the rating from a place card in the list. Return 'N/A' if it has no reviews yet."""
    try:
        raw = element.find_element(By.CSS_SELECTOR, "span.ZkP5Je").get_attribute("aria-label")
        return parse_rating(raw)
    except Exception:
        return "N/A"


print("✅ Functions defined")

## Cell 5 — Run the scraper
Opens Chrome, runs the search, visits each place, extracts the data, and collects it into `results`.

In [ ]:
# ============================================================
#  Self-test (runs without Chrome — under 1 second)
#  Proves the parsing works for every country/language, independent of keyword or city.
#  If you see "✅ ALL PASSED", the parsing logic is ready for any keyword/city.
# ============================================================

ICON = ""   # a real map-pin icon char (Private Use Area) that used to render as a box □

address_cases = [
    # (description, raw text from the page, expected clean result)
    ("Japan  (icon + newline + 〒 + hyphen)", ICON + "\nJapan, 〒150-0041 Tokyo, Shibuya",
     "Japan, 〒150-0041 Tokyo, Shibuya"),
    ("Thailand",                  ICON + "\n123 Sukhumvit Rd, Khlong Toei, Bangkok 10110",
     "123 Sukhumvit Rd, Khlong Toei, Bangkok 10110"),
    ("France  (accented chars)",  ICON + "\n12 Rue de l'Hôtel de Ville, 75004 Paris",
     "12 Rue de l'Hôtel de Ville, 75004 Paris"),
    ("USA     (stray emoji pin)", ICON + "\n\U0001f4cd 350 5th Ave, New York, NY 10118",
     "350 5th Ave, New York, NY 10118"),
    ("Korea   (Hangul + star)",   ICON + "\n★ 30 Eulji-ro, Jung-gu, Seoul",
     "30 Eulji-ro, Jung-gu, Seoul"),
    ("emoji + variation selector", "☺️ 1 Hacker Way, Menlo Park, CA",
     "1 Hacker Way, Menlo Park, CA"),
]

phone_cases = [
    ("Phone: +81 3-6380-4030", "+81 3-6380-4030"),
    ("Telefon: +49 30 123456", "+49 30 123456"),
    ("Téléphone: +33 1 53 01", "+33 1 53 01"),
]

rating_cases = [
    ("4.8 stars",             "4.8"),
    ("Rated 4.0 out of 5",    "4.0"),
    ("3.5 stars 120 reviews", "3.5"),
    ("No reviews yet",        "N/A"),
]

passed = failed = 0

def check(label, got, expect):
    global passed, failed
    ok = got == expect
    passed += ok; failed += (not ok)
    print(f"  {'✅' if ok else '❌'} {label}")
    if not ok:
        print(f"      expected: {expect!r}")
        print(f"      got     : {got!r}")

print("── address (clean_address_text) ──")
for label, raw, expect in address_cases:
    got = clean_address_text(raw)
    check(label, got, expect)
    # extra guard: the first character must no longer be a junk/icon character
    if got and ord(got[0]) >= 0xE000:
        print(f"      ⚠️ first char is still special: U+{ord(got[0]):04X}")

print("── phone (clean_phone_label) ──")
for raw, expect in phone_cases:
    check(repr(raw), clean_phone_label(raw), expect)

print("── rating (parse_rating) ──")
for raw, expect in rating_cases:
    check(repr(raw), parse_rating(raw), expect)

print("─" * 45)
if failed == 0:
    print(f"✅ ALL PASSED — {passed} cases. Parsing works for any keyword/city.")
else:
    print(f"❌ {failed} case(s) failed ({passed} passed) — see details above.")

## Cell 6 — Export to Excel

In [ ]:
if not results:
    print("⚠️ Nothing to save — check that the scraper ran successfully in Cell 5")
else:
    df = pd.DataFrame(results)

    # Preview the first 5 rows
    print(f"📊 Preview ({len(df)} places):")
    display(df.head())

    # Save to Excel
    df.to_excel(output_filename, index=False)
    print(f"\n💾 Saved: {output_filename}")

## Cell 6 — บันทึกผลลัพธ์เป็น Excel

In [ ]:
if not results:
    print("⚠️ ไม่มีข้อมูลให้บันทึก — ตรวจสอบว่า scraper รันสำเร็จใน Cell 5")
else:
    df = pd.DataFrame(results)

    # แสดงตัวอย่างข้อมูล 5 แถวแรก
    print(f"📊 ตัวอย่างข้อมูล ({len(df)} ร้าน):")
    display(df.head())

    # บันทึก Excel
    df.to_excel(output_filename, index=False)
    print(f"\n💾 บันทึกไฟล์สำเร็จ: {output_filename}")